# 05 · 전체 모델 비교와 논문용 결과 정리

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lab-axis/Wind-Power-Forecasting/blob/main/notebooks/05_comparison_and_research_limits.ipynb)

모든 표·그림은 동일한 새 실행의 검증된 저장 예측을 사용합니다. 본문 배치는 표준 모델 중심으로 간추리되 전체 15개 모델 결과와 확률 비교를 남깁니다. 표시 모델을 간추리는 것은 편집 결정이며 불리한 결과를 제거하거나 실험 전에 정한 것처럼 소급하는 절차가 아닙니다.

In [ ]:
from pathlib import Path
import os, sys

# Google Colab 자동 감지 및 환경 설정
if 'google.colab' in sys.modules or (os.path.exists('/content') and not (Path.cwd() / "configs/base_config.yaml").exists()):
    repo_dir = Path('/content/Wind-Power-Forecasting')
    if not repo_dir.exists():
        print("Google Colab 환경 감지: Wind-Power-Forecasting 저장소를 클론합니다...")
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/lab-axis/Wind-Power-Forecasting.git", str(repo_dir)], check=True)
        print("필수 의존 패키지를 설치합니다 (colab/requirements-colab.txt)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo_dir / "colab/requirements-colab.txt")], check=True)
    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))
    try:
        get_ipython().run_line_magic('cd', str(repo_dir))
    except Exception:
        pass

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/base_config.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
runtime = ROOT / ".experiment_archive/python_runtime"
if runtime.exists(): sys.path.insert(0, str(runtime))
from IPython.display import display
from src.workflows import research as wf
%matplotlib inline

In [ ]:
wf.run_status()

## 1. 공통 시점 확인

모든 모델·시드의 target_time, issue_time, y_true를 대조합니다. ARIMA·Persistence에도 동일한 공통 표본을 적용합니다. 데이터 누락이나 horizon 차이를 행 순서만으로 맞추지 않습니다.

In [ ]:
wf.verify_common_samples()

## 2. 본문 표: 5개 모델, 대표 3개 시계

**EMFN + Naive Persistence + ARIMA + LSTM + 일반 LightGBM(24h)**. LightGBM 계열은 본문에서 한 모델만 사용합니다. CRPS/Brier/RMSE/MAE/AUPRC를 나열하고 +1h/+6h/+24h를 표로, 전체 시계를 곡선으로 제시합니다. N/A는 점출력 모델에 예측 확률이 없다는 뜻입니다.

In [ ]:
wf.display_scores(models=wf.MAIN, horizons=(1,6,24))

In [ ]:
wf.plot_horizons(models=wf.MAIN);

## 3. 확률 베이스라인과 보정 한계

ARIMA, 분위수 LightGBM, 허들 분위수 LightGBM을 비교합니다. 기존 분위수·허들 모델의 수치가 유리하거나 불리해도 보존합니다. EMFN 대 점예측 모델의 CRPS 비교만으로 확률 모델 일반에 대한 우월성을 주장하지 않습니다. 명목 구간 포함률·구간 폭과 Brier를 같이 해석합니다. 점질량이 있는 중앙 예측구간은 보정된 분포에서도 포함률이 정확히 90%가 아닐 수 있습니다.

In [ ]:
probabilistic = ["EMFN (Proposed)", "ARIMA", "Quantile LightGBM (24h matched)", "Hurdle Quantile LightGBM (24h matched)"]
wf.display_scores(models=probabilistic, horizons=None)

In [ ]:
wf.plot_probability_comparison();

## 4. 검증 구간 결과와 전체 지표

검증 결과는 모델 선택에 사용한 구간의 점수이지 독립 성능이 아닙니다. 아래 변수에는 본문에서 생략한 모델·시계·전체 계산 지표가 모두 남습니다. raw 물리 위반율은 clipping 전 출력, 점오차는 bounded 출력에 대한 값입니다. 0에 가까운 발전량 때문에 MAPE가 불안정할 수 있으므로 보조 진단으로만 봅니다.

In [ ]:
wf.display_scores(split="validation", models=wf.MAIN, horizons=(1,6,24))

In [ ]:
full_results = wf.metrics_table(full=True, horizons=None)
display(full_results)
print("Full metric rows:", len(full_results))

## 5. 논문에서 남겨야 할 한계와 다음 단계

1. 2025년은 반복 열람한 개발 벤치마크. 2026년은 기술통계 열람 이력이 있으나 성능 선택에는 미사용이며 아직 평가하지 않았습니다.
2. 단위·1시간 집계 경계·계량 경계·관측 전달 지연의 기관 확인이 남습니다. 관측소 풍속은 터빈 허브 높이 풍속과 같다고 가정하지 않습니다.
3. 서로 다른 입력 예산(24h/168h/history/calendar), 학습 손실·예산을 명시합니다. Prophet의 train-only 달력 예측은 최근 상태를 반영하는 rolling-origin 모델과 정보 사용이 다릅니다.
4. 시드 SD는 학습 변동입니다. 기존 실행의 paired block 구간을 새 실행의 검정 결과로 재사용하지 않습니다.
5. 다중 시드 구조 어블레이션, 기상 변수군 기여, 최종 설계 고정 후 후속 기간 평가가 남습니다. 예측 정보 기여를 인과 효과로 바꾸어 주장하지 않습니다.

재현 추적: `reports/all_baselines_manifest.json` → `models/runs/<run_id>/source.zip`, 가중치, epoch 이력, validation/test 예측 → `reports/all_baselines_validation.json` → 최신 CSV. 모든 상세 로직은 `.py`에서 관리하며 노트북에는 중복 학습 구현을 두지 않습니다.